In [2]:
import os
import sklearn
import requests
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.preprocessing import StandardScaler
import itertools
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import CountVectorizer

In [3]:
# Bag of words (CountVectorizaer)
# tf-idf
# 3rd party
# put them into categories
# Get the data
# Clustering algorithms 9plot on graph) E.g, k-means
#  Use the clustering to put them into categories
with open("home_data.json", "r") as data:
    home_data = json.load(data) 

user_preferences= [
      "Bicycle Storage",
      "Recessed dimmable lighting",
      "Living rooms with soaring 9ft 6in ceilings",
      "Complimentary WiFi in amenity spaces",
      "Pet-friendly with pet spa",
      "Flooring: Wood",
      "Electric car charging stations",
      "Keyless entry",
      "Backlit mirrors walk-in showers Moen fixtures",
      "Premium ENERGY STAR appliances",
      "Dogs Allowed",
      "Cats Allowed"
]
# preferences = []
# user_preferences = {
#     "cooling": "Central Air",
#     "flooring": "Hardwood",
#     "property_type": "Apartment",
#     "year_built": 2015,
#     "pet_friendly": True,
#     "parking": "Garage",
#     "Furnished": True,
#     "neighborhood_score": 8.5,
#     "walkability_score": 7.2,
#     "amenities": ["gym", "pool", "rooftop lounge"],
#     "public_transport_proximity": "0.3 miles",
#     "price_trend": "stable"
# }

# user_preferences = list(user_preferences.values())
# print(user_preferences)

# ----STEPS TO ENCIDE THE PREFERENCES-----
    # Use bag of words or sentence embedding for the amneties
# Sample user liked houses
Liked_house_id = ['6786270504', '6541999527', "5332041691"]

In [4]:
# Get the data needed
data_needed = {}
for i in range(0,100):
    details = []
    if home_data["data"]["results"][i]["details"] != None:
        details = home_data["data"]["results"][i]["details"][0]["text"]
    # get the data needed from the API
    id = home_data["data"]["results"][i]["property_id"]
    data_needed[id] = details
with open("cur_data.json", "w") as outfile:
    json.dump(data_needed, outfile, indent=3)

In [5]:
# TO-DO
    # Apply weights

In [6]:
House_data = data_needed
User_prefrences = user_preferences
house_ids = list(House_data.keys()) #Get house IDs
house_features = list(House_data.values()) #Get house features

# Combine user preferences and house data values. Now using house_features
all_data = [" ".join(User_prefrences)] + [" ".join(house) for house in house_features]

# Create and fit the CountVectorizer
# vectorizer = CountVectorizer()
vectorizer = TfidfVectorizer()
vectorizer.fit(all_data)

# Transform user preferences and house data
user_vector = vectorizer.transform([" ".join(User_prefrences)])
house_vectors = vectorizer.transform([" ".join(house) for house in house_features])

# Calculate cosine similarity
similarity_scores = cosine_similarity(user_vector, house_vectors)[0]

# Create a list of tuples (similarity_score, house_id) using house_ids
ranked_houses = sorted(zip(similarity_scores, house_ids), reverse=True)

# Print the ranked houses
for score, house_id in ranked_houses:
    print(f"House ID: {house_id}, Similarity Score: {score}")

ranked_houses_dict = {house_id: score for score, house_id in ranked_houses}

# Print the dictionary
# print("\nRanked Houses Dictionary:")
# print(ranked_houses_dict)

House ID: 5508300611, Similarity Score: 1.0000000000000004
House ID: 9884543104, Similarity Score: 0.21472148829549756
House ID: 6541999527, Similarity Score: 0.20921703558769839
House ID: 9356931592, Similarity Score: 0.19432781255175932
House ID: 6463335616, Similarity Score: 0.19162975377678748
House ID: 9200901531, Similarity Score: 0.19006438985643673
House ID: 6908347987, Similarity Score: 0.1840080253140028
House ID: 9750192392, Similarity Score: 0.18270446043474609
House ID: 5152321841, Similarity Score: 0.18113893716668847
House ID: 6274747906, Similarity Score: 0.17374643595476588
House ID: 9734095827, Similarity Score: 0.16603668644108693
House ID: 9709087888, Similarity Score: 0.15049268085609463
House ID: 5670416449, Similarity Score: 0.14904090381370755
House ID: 9015515475, Similarity Score: 0.14733331997745658
House ID: 9942798138, Similarity Score: 0.14700345540287954
House ID: 6786270504, Similarity Score: 0.14606164805140048
House ID: 6457372760, Similarity Score: 0.

In [7]:
# Get indices, handling potential missing IDs
liked_house_indices = [i for i, house_id in enumerate(house_ids) if house_id in Liked_house_id]
print(liked_house_indices)

if liked_house_indices:
    average_similarity = np.mean(similarity_scores[liked_house_indices]) #Calculates the average similarity score 
else:
    average_similarity = 0
print(similarity_scores[liked_house_indices])
    

# Use NumPy for filtering
recommended_mask = (similarity_scores > average_similarity) & ~np.isin(house_ids, Liked_house_id)
recommended_houses = list(zip(np.array(house_ids)[recommended_mask], similarity_scores[recommended_mask]))

if recommended_houses:
    print("Recommended Houses (above average similarity):")
    for house_id, score in recommended_houses:
        print(f"House ID: {house_id}, Similarity Score: {score}")
else:
    print("No houses found with similarity scores greater than the average similarity of liked houses.")

#Alternative way to recommend top N houses
N = 3 # Number of top recommendations
top_n_recommendations = [(house_id, score) for house_id, score in ranked_houses if house_id not in Liked_house_id][:N]

if top_n_recommendations:
  print(f"\nTop {N} Recommended Houses (excluding liked houses):")
  for score, house_id in top_n_recommendations:
      print(f"House ID: {house_id}, Similarity Score: {score}")
else:
    print("No other houses to recommend.")

[1, 2, 4]
[0.14606165 0.20921704 0.08847849]
Recommended Houses (above average similarity):
House ID: 5508300611, Similarity Score: 1.0000000000000004
House ID: 5670416449, Similarity Score: 0.14904090381370755
House ID: 6908347987, Similarity Score: 0.1840080253140028
House ID: 9200901531, Similarity Score: 0.19006438985643673
House ID: 6463335616, Similarity Score: 0.19162975377678748
House ID: 9734095827, Similarity Score: 0.16603668644108693
House ID: 6274747906, Similarity Score: 0.17374643595476588
House ID: 5152321841, Similarity Score: 0.18113893716668847
House ID: 9356931592, Similarity Score: 0.19432781255175932
House ID: 9750192392, Similarity Score: 0.18270446043474609
House ID: 9884543104, Similarity Score: 0.21472148829549756
House ID: 9709087888, Similarity Score: 0.15049268085609463

Top 3 Recommended Houses (excluding liked houses):
House ID: 5508300611, Similarity Score: 1.0000000000000004
House ID: 9884543104, Similarity Score: 0.21472148829549756
House ID: 654199952